In [ ]:
!sudo apt install tesseract-ocr
!sudo apt install tesseract-ocr-eng tesseract-ocr-ben

**the qwen 2.5 3B is need to be quantized.**

In [ ]:
# It's highly recommanded to use `[decord]` feature for faster video loading.
!pip install qwen-vl-utils[decord]==0.0.8


In [ ]:
!pip install --upgrade autoawq

In [9]:
# pip install bitsandbytes accelerpip install flash-attn --no-build-isolationate
!pip install flash-attn --no-build-isolation

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

In [16]:
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
import torch
import os
from PIL import Image
torch.cuda.empty_cache()
# os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
# default: Load the model on the available device(s)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2.5-VL-3B-Instruct-AWQ",   torch_dtype=torch.bfloat16,
        attn_implementation="flash_attention_2",
    device_map="cuda:0"
)
img = Image.open("/kaggle/input/trying/ff.png")
# Resize to smaller dimensions
# img = img.resize((512, 512))

# We recommend enabling flash_attention_2 for better acceleration and memory saving, especially in multi-image and video scenarios.
# model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
#     "Qwen/Qwen2.5-VL-3B-Instruct",
#     torch_dtype=torch.bfloat16,
#     attn_implementation="flash_attention_2",
#     device_map="auto",
# )

# default processer
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct")

# The default range for the number of visual tokens per image in the model is 4-16384.
# You can set min_pixels and max_pixels according to your needs, such as a token range of 256-1280, to balance performance and cost.
# min_pixels = 256*28*28
# max_pixels = 1280*28*28
# processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct", min_pixels=min_pixels, max_pixels=max_pixels)

messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": img,
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

# Preparation for inference
text = processor.apply_chat_template(
    messages, tokenize=False, add_generation_prompt=True
)
image_inputs, video_inputs = process_vision_info(messages)
inputs = processor(
    text=[text],
    images=image_inputs,
    videos=video_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda" )

# Inference: Generation of the output
generated_ids = model.generate(**inputs, max_new_tokens=64)
generated_ids_trimmed = [
    out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)


We suggest you to set `torch_dtype=torch.float16` for better efficiency with AWQ.
Some weights of Qwen2_5_VLForConditionalGeneration were not initialized from the model checkpoint at Qwen/Qwen2.5-VL-3B-Instruct-AWQ and are newly initialized: ['lm_head.qweight', 'lm_head.qzeros', 'lm_head.scales']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


NameError: name 'apply_rotary_emb' is not defined

* Moondream

In [ ]:
!pip install pyvips

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from PIL import Image

image = Image.open('/kaggle/input/nope-1/493411178_1490357505276581_7381946568352867287_n.jpg')
model = AutoModelForCausalLM.from_pretrained(
    "vikhyatk/moondream2",
    revision="2025-04-14",
    trust_remote_code=True,
    # Uncomment to run on GPU.
    device_map={"": "cuda"}
)

# Captioning
print("Short caption:")
print(model.caption(image, length="short")["caption"])

print("\nNormal caption:")
for t in model.caption(image, length="normal", stream=True)["caption"]:
    # Streaming generation example, supported for caption() and detect()
    print(t, end="", flush=True)
print(model.caption(image, length="normal"))

# Visual Querying
print("\nVisual query: 'How many people are in the image?'")
print(model.query(image, "How many people are in the image?")["answer"])

# Object Detection
print("\nObject detection: 'face'")
objects = model.detect(image, "face")["objects"]
print(f"Found {len(objects)} face(s)")

# Pointing
print("\nPointing: 'person'")
points = model.point(image, "person")["points"]
print(f"Found {len(points)} person(s)")


In [ ]:
import easyocr
import cv2
import pytesseract
import os
import matplotlib.pyplot as plt
from PIL import Image # pytesseract often works well with PIL images
# Specify the path to your image file
image_path = '/kaggle/input/trying/ff.png' # Replace with your image path

# Check if the image file exists
if not os.path.exists(image_path):
    print(f"Error: Image file not found at {image_path}")
else:
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not load image from {image_path}")
    else:
        # --- EasyOCR for Detection ---
        print("Initializing EasyOCR reader for detection...")
        try:
            # Use EasyOCR for detection only
            # We still need language models even just for robust detection
            reader = easyocr.Reader(['en', 'bn'], gpu=True) # Attempt GPU
        except:
            reader = easyocr.Reader(['en', 'bn'], gpu=False) # Fallback to CPU
        print("EasyOCR Reader initialized.")

        # Get bounding boxes from EasyOCR
        # detail=1 gives bounding box, text, confidence
        # We are primarily interested in bbox here
        detection_results = reader.readtext(img, detail=1)

        print(f"Detected {len(detection_results)} potential text regions using EasyOCR.")

        # --- Pytesseract for Recognition on each region ---
        print("\n--- Extracting text from each region using Pytesseract ---")
        extracted_texts = []

        for (bbox, text, confidence) in detection_results:
            # Get the coordinates of the bounding box
            # bbox points are [[x1, y1], [x2, y2], [x3, y3], [x4, y4]]
            # We need top-left (x1, y1) and bottom-right (x3, y3) for cropping
            x_coords = [point[0] for point in bbox]
            y_coords = [point[1] for point in bbox]

            # Find min/max to get the bounding rectangle
            x1, y1 = int(min(x_coords)), int(min(y_coords))
            x2, y2 = int(max(x_coords)), int(max(y_coords))

            # Add a small padding to the bounding box (optional, can help)
            padding = 5
            x1 = max(0, x1 - padding)
            y1 = max(0, y1 - padding)
            x2 = min(img.shape[1], x2 + padding)
            y2 = min(img.shape[0], y2 + padding)


            # Crop the image region using OpenCV array slicing (NumPy arrays)
            cropped_img = img[y1:y2, x1:x2]

            # --- Optional: Preprocess the cropped image for Tesseract ---
            # Tesseract often works well with binary images.
            # This might need tuning depending on your specific cropped regions.
            # cropped_gray = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2GRAY)
            # cropped_thresh = cv2.threshold(cropped_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
            # image_to_feed_to_tesseract = cropped_thresh # Use the thresholded image
            image_to_feed_to_tesseract = cropped_img # Use the original cropped color/grayscale image


            # Convert the OpenCV image (NumPy array) to a PIL Image
            # Pytesseract's image_to_string often works better with PIL images
            pil_img = Image.fromarray(image_to_feed_to_tesseract)
            img_rgb = cv2.cvtColor(image_to_feed_to_tesseract, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(10, 10)) # Adjust figure size as needed
            plt.imshow(img_rgb)
            plt.title("region")
            plt.axis('off') # Hide axes
            plt.show() 
            # Use Pytesseract to extract text from the cropped region
            # lang='eng+ben' specifies English and Bengali language packs
            try:
                text_from_region = pytesseract.image_to_string(pil_img, lang='ben')

                # Clean up extracted text (remove extra newlines/spaces)
                cleaned_text = text_from_region.strip()

                if cleaned_text: # Only print if text was found
                    print(f"Region ({x1},{y1}) to ({x2},{y2}):")
                    print(f"  '{cleaned_text}'")
                    extracted_texts.append(cleaned_text)
                # else:
                    # print(f"Region ({x1},{y1}) to ({x2},{y2}): No text detected by Pytesseract")


            except pytesseract.TesseractNotFoundError:
                print("\nError: Tesseract executable not found.")
                print("Please install Tesseract OCR engine and make sure it's in your system's PATH or update the pytesseract.pytesseract.tesseract_cmd variable in the script.")
                break # Exit the loop if Tesseract is not found
            except Exception as e:
                 print(f"An error occurred processing region ({x1},{y1}) to ({x2},{y2}): {e}")
                 # Optional: Print the cropped image to inspect if an error occurs
                 # cv2.imwrite(f"error_region_{x1}_{y1}.png", cropped_img)


        print("\n--- Combined Text from all regions (Pytesseract) ---")
        print("\n".join(extracted_texts))

In [ ]:
import easyocr
import cv2
import matplotlib.pyplot as plt
import os

# --- Configuration ---
# Specify the path to your image file
# You can use one of the images you uploaded, e.g., 'image_6a8af0.jpg'
image_path = '/kaggle/input/trying/ff.png' # Replace with your image path

# Specify languages for EasyOCR detection (even for just detection,
# language models influence what is considered text)
languages = ['en', 'bn']

# Color for the bounding boxes (BGR format)
box_color = (0, 255, 0) # Green color

# Thickness of the bounding box lines
box_thickness = 2

# --- Code ---
# Check if the image file exists
if not os.path.exists(image_path):
    print(f"Error: Image file not found at {image_path}")
else:
    # Load the image using OpenCV
    try:
        img = cv2.imread(image_path)
        if img is None:
            print(f"Error: Could not load image from {image_path}")
        else:
            # Initialize the OCR reader
            print("Initializing EasyOCR reader (downloading models if necessary)...")
            try:
                 # Attempt to use GPU, fallback to CPU if error occurs
                reader = easyocr.Reader(languages, gpu=True)
            except:
                 # Fallback to CPU
                reader = easyocr.Reader(languages, gpu=False)
            print("Reader initialized.")

            # Perform text detection (this also does recognition, but we only need the boxes)
            print(f"Detecting text regions in {image_path}...")
            # detail=1 gives bounding box, text and confidence
            # detail=0 gives only the text
            # We need detail=1 for bounding boxes
            results = reader.readtext(img, detail=1)

            print(f"Found {len(results)} text regions.")

            # Draw bounding boxes on the image
            for (bbox, text, confidence) in results:
                # bbox is a list of 4 points [[x1, y1], [x2, y2], [x3, y3], [x4, y4]]
                # For drawing a rectangle, we typically need top-left (x1, y1) and bottom-right (x3, y3)
                # Convert points to integers
                top_left = tuple(map(int, bbox[0]))
                bottom_right = tuple(map(int, bbox[2]))

                # Draw the rectangle on the image
                cv2.rectangle(img, top_left, bottom_right, box_color, box_thickness)

                # Optional: Put the confidence score near the box
                # cv2.putText(img, f"{confidence:.2f}", (top_left[0], top_left[1] - 10),
                #             cv2.FONT_HERSHEY_SIMPLEX, 0.5, box_color, 1)


            # Display the image with bounding boxes
            # OpenCV's imshow works best in environments where it can open windows
            # In some environments (like notebooks), you might need matplotlib

            # --- Displaying the image ---
            # Method 1: Using OpenCV's imshow (works in standard Python scripts with GUI)
            # cv2.imshow("Text Detection", img)
            # cv2.waitKey(0) # Wait indefinitely until a key is pressed
            # cv2.destroyAllWindows() # Close all OpenCV windows

            # Method 2: Using Matplotlib (works well in notebooks or environments without a direct GUI window)
            # Convert color from BGR (OpenCV) to RGB (Matplotlib)
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            plt.figure(figsize=(10, 10)) # Adjust figure size as needed
            plt.imshow(img_rgb)
            plt.title("Text Detection with Bounding Boxes")
            plt.axis('off') # Hide axes
            plt.show() # Display the plot

    except Exception as e:
        print(f"An error occurred: {e}")

In [ ]:
# Specify the path to your image file
 # Replace with your image path

# Check if the image file exists
def easy_pytesseract(image_path):
    if not os.path.exists(image_path):
        print(f"Error: Image file not found at {image_path}")
    else:
        img = cv2.imread(image_path)
        if img is None:
            print(f"Error: Could not load image from {image_path}")
        else:
            # --- EasyOCR for Detection ---
            print("Initializing EasyOCR reader for detection...")
            try:
                # Use EasyOCR for detection only
                # We still need language models even just for robust detection
                reader = easyocr.Reader(['en', 'bn'], gpu=True) # Attempt GPU
            except:
                reader = easyocr.Reader(['en', 'bn'], gpu=False) # Fallback to CPU
            print("EasyOCR Reader initialized.")
    
            # Get bounding boxes from EasyOCR
            # detail=1 gives bounding box, text, confidence
            # We are primarily interested in bbox here
            detection_results = reader.readtext(img, detail=1)
    
            print(f"Detected {len(detection_results)} potential text regions using EasyOCR.")
    
            # --- Pytesseract for Recognition on each region ---
            print("\n--- Extracting text from each region using Pytesseract ---")
            extracted_texts = []
    
            for (bbox, text, confidence) in detection_results:
                # Get the coordinates of the bounding box
                # bbox points are [[x1, y1], [x2, y2], [x3, y3], [x4, y4]]
                # We need top-left (x1, y1) and bottom-right (x3, y3) for cropping
                x_coords = [point[0] for point in bbox]
                y_coords = [point[1] for point in bbox]
    
                # Find min/max to get the bounding rectangle
                x1, y1 = int(min(x_coords)), int(min(y_coords))
                x2, y2 = int(max(x_coords)), int(max(y_coords))
    
                # Add a small padding to the bounding box (optional, can help)
                padding = 5
                x1 = max(0, x1 - padding)
                y1 = max(0, y1 - padding)
                x2 = min(img.shape[1], x2 + padding)
                y2 = min(img.shape[0], y2 + padding)
    
    
                # Crop the image region using OpenCV array slicing (NumPy arrays)
                cropped_img = img[y1:y2, x1:x2]
    
                # --- Optional: Preprocess the cropped image for Tesseract ---
                # Tesseract often works well with binary images.
                # This might need tuning depending on your specific cropped regions.
                # cropped_gray = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2GRAY)
                # cropped_thresh = cv2.threshold(cropped_gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1]
                # image_to_feed_to_tesseract = cropped_thresh # Use the thresholded image
                image_to_feed_to_tesseract = cropped_img # Use the original cropped color/grayscale image
    

                
                # Convert the OpenCV image (NumPy array) to a PIL Image
                # Pytesseract's image_to_string often works better with PIL images
                pil_img = Image.fromarray(image_to_feed_to_tesseract)
    
                
                # Use Pytesseract to extract text from the cropped region
                # lang='eng+ben' specifies English and Bengali language packs
                try:
                    text_from_region = pytesseract.image_to_string(pil_img, lang='ben')
    
                    # Clean up extracted text (remove extra newlines/spaces)
                    cleaned_text = text_from_region.strip()
    
                    if cleaned_text: # Only print if text was found
                        print(f"Region ({x1},{y1}) to ({x2},{y2}):")
                        print(f"  '{cleaned_text}'")
                        extracted_texts.append(cleaned_text)
                    # else:
                        # print(f"Region ({x1},{y1}) to ({x2},{y2}): No text detected by Pytesseract")
    
    
                except pytesseract.TesseractNotFoundError:
                    print("\nError: Tesseract executable not found.")
                    print("Please install Tesseract OCR engine and make sure it's in your system's PATH or update the pytesseract.pytesseract.tesseract_cmd variable in the script.")
                    break # Exit the loop if Tesseract is not found
                except Exception as e:
                     print(f"An error occurred processing region ({x1},{y1}) to ({x2},{y2}): {e}")
                     # Optional: Print the cropped image to inspect if an error occurs
                     # cv2.imwrite(f"error_region_{x1}_{y1}.png", cropped_img)
    
    
            print("\n--- Combined Text from all regions (Pytesseract) ---")
            print("\n".join(extracted_texts))
image_path = '/kaggle/input/trying/ff.png'

easy_pytesseract(image_path)